In [ ]:
import os
import itertools
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle
from statsmodels.stats.multitest import fdrcorrection

RESULTS = "../4_results"
ATLAS   = "Shen"          # "Shen" (268 ROIs) or "Schaefer" (Schaefer 200 + 8 subcortical = 208 ROIs); must match b
ALPHA   = 0.05            # the feature-selection alpha of the classification run to read

CLS_DIR = os.path.join(RESULTS, f"2_classification_{ATLAS}_{ALPHA:g}")   # input: b_Classification output
NET_DIR = os.path.join(RESULTS, f"3_network_{ATLAS}_{ALPHA:g}")          # output: everything this notebook makes
FIG_DIR = os.path.join(NET_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Shen: network label per ROI from this CSV (columns ROI (1-based), Network, ...)
ATLAS_CSV = "../3_atlases/Shen/shen268_with_schaefer200_networks.csv"
# Schaefer: ROIs 1-200 = Schaefer 200 parcels (17 networks, labels from nilearn), 201-208 = the 8 subcortical ROIs
N_SUBCORTICAL     = 8
SUBCORTICAL_NAMES = ["L_amygdala", "L_basal_ganglia", "L_hippocampus", "L_thalamus",
                     "R_amygdala", "R_basal_ganglia", "R_hippocampus", "R_thalamus"]   # ROIs 201-208
N_ROIS_ATLAS = {"Shen": 268, "Schaefer": 208}[ATLAS]

# The four models compared (each trained on all subjects of its dataset)
FEATURE_FILES = {
    "PupilRest":  os.path.join(CLS_DIR, "4_EEG_Rest_from_Pupil_Rest", "EEG_Rest_from_Pupil_Rest_features.csv"),
    "EEGRest":    os.path.join(CLS_DIR, "5_Pupil_Rest_from_EEG_Rest", "Pupil_Rest_from_EEG_Rest_features.csv"),
    "PupilStory": os.path.join(CLS_DIR, "7_Pupil_Rest_from_Pupil_Story", "Pupil_Rest_from_Pupil_Story_features.csv"),
    "Movies":     os.path.join(CLS_DIR, "1_Movies_within", "Movies_features.csv"),
}
FOUR_WAY = "_".join(FEATURE_FILES)                     # PupilRest_EEGRest_PupilStory_Movies

N_PERM_OVERLAP = 1000      # overlap null (same as before; it's fast now, so 10000 is fine too)
N_PERM_NETWORK = 10000     # network-matrix null
FDR            = True      # stars on the network matrix use FDR-corrected p-values
SEED           = 42

# number of ROIs in this atlas: from the CPM export of the classification run if available
_pairs = os.path.join(RESULTS, "CPM", f"feature_roi_pairs_{ATLAS}.csv")
N_ROIS = int(pd.read_csv(_pairs)["ROI2"].max()) if os.path.exists(_pairs) else N_ROIS_ATLAS
print(f"Atlas {ATLAS} ({N_ROIS} ROIs), alpha {ALPHA:g}\n  reading  {CLS_DIR}\n  writing  {NET_DIR}")
missing = [p for p in FEATURE_FILES.values() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Run b_Classification with ATLAS = '{ATLAS}', ALPHA = {ALPHA} first. Missing: {missing}")

# Sign-Consistent Model Overlap

In [ ]:
def load_signed_edges(path):
    """{(ROI_lo, ROI_hi): 'pos'/'neg'} from a features CSV (ROI1, ROI2, feature_weight)."""
    df = pd.read_csv(path)
    w = df["feature_weight"].to_numpy(dtype=float)
    r1, r2 = df["ROI1"].astype(int).to_numpy(), df["ROI2"].astype(int).to_numpy()
    out = {}
    for a, b, x in zip(np.minimum(r1, r2), np.maximum(r1, r2), w):
        if x > 0:
            out[(int(a), int(b))] = "pos"
        elif x < 0:
            out[(int(a), int(b))] = "neg"
    return out


def test_nway_sign_overlap(feature_files, k, n_permutations=1000, n_rois=N_ROIS, output_dir=NET_DIR, seed=SEED):
    """
    For every combination of k models: edges selected by all of them with the same sign, and a null where
    each model's positive and negative edge counts are redrawn at random from all n_rois*(n_rois-1)/2 edges.
    p = (1 + #null >= observed) / (n_permutations + 1).  k = 1 writes a per-model feature summary.
    """
    os.makedirs(output_dir, exist_ok=True)
    feature_sets = {name: load_signed_edges(path) for name, path in feature_files.items()}
    n_edges_total = n_rois * (n_rois - 1) // 2
    rng = np.random.default_rng(seed)

    if k == 1:
        rows = []
        for model, feats in feature_sets.items():
            n_pos = sum(s == "pos" for s in feats.values())
            n_neg = sum(s == "neg" for s in feats.values())
            rows.append({"Model": model, "Positive_Features": n_pos, "Negative_Features": n_neg,
                         "Total_Features": n_pos + n_neg})
            print(f"{model:11s} +{n_pos} / -{n_neg}")
        summary = pd.DataFrame(rows)
        summary.to_csv(os.path.join(output_dir, "feature_sign_summary_k1.csv"), index=False)
        return summary

    results = []
    for combo in combinations(feature_sets, k):
        combo_name = "_".join(combo)
        maps = [feature_sets[m] for m in combo]
        common = set.intersection(*[set(d) for d in maps])
        consistent = sorted(e for e in common if len({d[e] for d in maps}) == 1)
        observed = len(consistent)
        pd.DataFrame({"ROI1": [e[0] for e in consistent], "ROI2": [e[1] for e in consistent],
                      "Sign": [maps[0][e] for e in consistent]}).to_csv(
            os.path.join(output_dir, f"overlapping_features_{combo_name}.csv"), index=False)

        counts = [(sum(s == "pos" for s in d.values()), sum(s == "neg" for s in d.values())) for d in maps]
        null = np.empty(n_permutations, dtype=int)
        for i in range(n_permutations):
            pos_hits = np.zeros(n_edges_total, dtype=np.int8)
            neg_hits = np.zeros(n_edges_total, dtype=np.int8)
            for n_pos, n_neg in counts:
                idx = rng.choice(n_edges_total, size=n_pos + n_neg, replace=False)
                pos_hits[idx[:n_pos]] += 1
                neg_hits[idx[n_pos:]] += 1
            null[i] = np.count_nonzero(pos_hits == k) + np.count_nonzero(neg_hits == k)
        p_value = (1 + np.sum(null >= observed)) / (n_permutations + 1)
        np.save(os.path.join(output_dir, f"null_dist_{combo_name}.npy"), null)
        print(f"{combo_name:40s} overlap {observed:5d}   null mean {null.mean():7.1f}   p = {p_value:.6f}")
        results.append({"Combination": combo_name, "Set_Size": k, "Observed_Overlap": observed,
                        "Mean_Null_Overlap": null.mean(), "P_Value": p_value})

    result_df = pd.DataFrame(results)
    result_df.to_csv(os.path.join(output_dir, f"overlap_results_{k}way_by_model.csv"), index=False)
    return result_df

In [ ]:
overlap_results = {}
for k in (1, 2, 3, 4):
    print(f"\n--- {k}-way ---")
    overlap_results[k] = test_nway_sign_overlap(FEATURE_FILES, k=k, n_permutations=N_PERM_OVERLAP, n_rois=N_ROIS)
master = pd.concat([overlap_results[k] for k in (2, 3, 4)], ignore_index=True)
master.to_csv(os.path.join(NET_DIR, "master_overlap_summary.csv"), index=False)   # rewritten each run (no duplicates)
print(f"\nSaved overlap results to {NET_DIR}")

# Network Edge Matrix

In [ ]:
# ---- ROI lookup: {ROI number: (name, sub-network, network)} for the chosen atlas ----
SHORT = {"Control": "CON", "Dorsal Attention": "DAN", "Default": "DMN", "Salience": "SAL",
         "Somatomotor": "SM", "Visual": "VIS", "Limbic": "LIM", "Cerebellum": "CB", "Subcortical": "SB"}
# Schaefer only: the 17 subnetworks (grouped in the same order as ORDER) + Subcortical, and their parent networks
ORDER17 = ["ContA", "ContB", "ContC", "DorsAttnA", "DorsAttnB", "DefaultA", "DefaultB", "DefaultC",
           "TempPar", "SalVentAttnA", "SalVentAttnB", "SomMotA", "SomMotB", "VisCent", "VisPeri",
           "LimbicA", "LimbicB", "Subcortical"]
SHORT17 = {"ContA": "CON-A", "ContB": "CON-B", "ContC": "CON-C", "DorsAttnA": "DAN-A",
           "DorsAttnB": "DAN-B", "DefaultA": "DMN-A", "DefaultB": "DMN-B", "DefaultC": "DMN-C",
           "TempPar": "TP", "SalVentAttnA": "SAL-A", "SalVentAttnB": "SAL-B", "SomMotA": "SM-A",
           "SomMotB": "SM-B", "VisCent": "VIS-C", "VisPeri": "VIS-P", "LimbicA": "LIM-A",
           "LimbicB": "LIM-B", "Subcortical": "SB"}
SCHAEFER17_PARENT = {"TempPar": "Default", "Default": "Default", "Cont": "Control",
                     "DorsAttn": "Dorsal Attention", "SalVentAttn": "Salience",
                     "SomMot": "Somatomotor", "Vis": "Visual", "Limbic": "Limbic"}


def shen_lookup(atlas_csv=ATLAS_CSV):
    atlas = pd.read_csv(atlas_csv)
    return {int(r): (f"Shen_{int(r)}", net, net) for r, net in zip(atlas["ROI"], atlas["Network"])}


def schaefer_lookup(n_subcortical=N_SUBCORTICAL, subcortical_names=SUBCORTICAL_NAMES):
    import nilearn.datasets                             # downloads the Schaefer labels once (cached by nilearn)
    labels = nilearn.datasets.fetch_atlas_schaefer_2018(n_rois=200, yeo_networks=17, resolution_mm=2).labels
    labels = [l.decode() if isinstance(l, bytes) else str(l) for l in labels]
    if labels[0].lower().startswith("background"):
        labels = labels[1:]
    if subcortical_names is None:
        subcortical_names = [f"Subcortical_{i}" for i in range(1, n_subcortical + 1)]
    lookup = {}
    for roi, lab in enumerate(labels, start=1):
        sub = lab.split("_")[2]                                 # "17Networks_LH_DefaultA_pCunPCC_1" -> "DefaultA"
        parent = next((p for prefix, p in SCHAEFER17_PARENT.items() if sub.startswith(prefix)), "Unknown")
        lookup[roi] = (lab, sub, parent)
    for i, name in enumerate(subcortical_names, start=len(labels) + 1):
        lookup[i] = (name, "Subcortical", "Subcortical")
    return lookup


if ATLAS == "Shen":
    LOOKUP = shen_lookup()
    ORDER = ["Control", "Dorsal Attention", "Default", "Salience", "Somatomotor", "Visual", "Limbic", "Cerebellum",
             "Subcortical"]
else:                                                  # no cerebellum: an empty grey CB row/column keeps the Shen layout
    LOOKUP = schaefer_lookup()
    ORDER = ["Control", "Dorsal Attention", "Default", "Salience", "Somatomotor", "Visual", "Limbic", "Subcortical"]
if len(LOOKUP) != N_ROIS:
    print(f"⚠️ The {ATLAS} label list has {len(LOOKUP)} ROIs but the FC data has {N_ROIS}. Check the ROI order/labels.")


def network_edge_matrix(feature_csv, collapse=True, order=None, short_names=None, lookup=LOOKUP,
                        n_perm=10000, null_model="roi_subset", include_within=False, fdr=False, alpha=0.05,
                        cmap="Greens", seed=42, save_path=None, pad_cerebellum=True):
    """
    Count selected edges per network pair, normalize by the number of possible edges,
    test each pair against a permutation null, and plot a lower-triangle heatmap with stars.

    collapse       : True  -> networks in ORDER (Shen: the atlas CSV's Network; Schaefer: 8 parent networks, DMN
                              incl. TempPar)
                     False -> Schaefer only: the 17 original subnetworks + SB (18 x 18 matrix)
    null_model     : "roi_subset" (default) - each permutation picks as many random ROIs as your edges
                     involve, then draws the same number of distinct edges among only those ROIs.
                     "edges" - draws the same number of edges from all ROI pairs in the atlas.
    pad_cerebellum : collapsed plot only, atlases without a cerebellum (Schaefer) - draw an empty, greyed-out CB
                     row/column between LIM and SB so the matrix has the same 9 x 9 layout as the Shen plots (display only: the statistics, FDR
                     correction and returned table are unchanged)
    Returns (results DataFrame, matplotlib Figure).
    """
    if order is None:
        order = ORDER if collapse else ORDER17
    if short_names is None:
        short_names = SHORT if collapse else SHORT17
    rng = np.random.default_rng(seed)
    n_net = len(order)

    roi_networks = {roi: (v[2] if collapse else v[1]) for roi, v in lookup.items()}
    missing = sorted(set(roi_networks.values()) - set(order))
    if missing:
        print(f"Networks in the atlas but not in order (their ROIs are ignored): {missing}")
    net_index = {name: k for k, name in enumerate(order)}
    roi_to_net = {roi: net_index[net] for roi, net in roi_networks.items() if net in net_index}

    edges = pd.read_csv(feature_csv).dropna(subset=["ROI1", "ROI2"])
    observed = np.zeros((n_net, n_net))
    n_edges, n_skipped = 0, 0
    used_rois = set()
    for r1, r2 in zip(edges["ROI1"].astype(int), edges["ROI2"].astype(int)):
        if r1 not in roi_to_net or r2 not in roi_to_net:
            n_skipped += 1
            continue
        a, b = roi_to_net[r1], roi_to_net[r2]
        observed[a, b] += 1
        if a != b:
            observed[b, a] += 1
        n_edges += 1
        used_rois.update((r1, r2))
    n_rois = len(used_rois)
    if n_skipped:
        print(f"Skipped {n_skipped} edge(s) touching ROIs outside {order}")

    sizes = np.bincount(list(roi_to_net.values()), minlength=n_net)
    possible = np.outer(sizes, sizes).astype(float)
    np.fill_diagonal(possible, sizes * (sizes - 1) / 2)
    with np.errstate(divide="ignore", invalid="ignore"):
        density = np.where(possible > 0, observed / possible, np.nan)

    nets = np.array(list(roi_to_net.values()))
    null = np.empty((n_perm, n_net * n_net))
    if null_model == "roi_subset":
        iu = np.triu_indices(n_rois, k=1)
        for k in range(n_perm):
            subset = nets[rng.choice(len(nets), size=n_rois, replace=False)]
            draw = rng.choice(len(iu[0]), size=n_edges, replace=False)
            a, b = subset[iu[0][draw]], subset[iu[1][draw]]
            null[k] = np.bincount(np.minimum(a, b) * n_net + np.maximum(a, b), minlength=n_net * n_net)
    elif null_model == "edges":
        iu = np.triu_indices(len(nets), k=1)
        pair_id = np.minimum(nets[iu[0]], nets[iu[1]]) * n_net + np.maximum(nets[iu[0]], nets[iu[1]])
        for k in range(n_perm):
            draw = rng.choice(len(pair_id), size=n_edges, replace=False)
            null[k] = np.bincount(pair_id[draw], minlength=n_net * n_net)
    else:
        raise ValueError('null_model must be "roi_subset" or "edges"')

    rows, cols = np.tril_indices(n_net, k=0 if include_within else -1)
    ids = cols * n_net + rows
    obs_vals = observed[rows, cols]
    p_raw = ((null[:, ids] >= obs_vals).sum(axis=0) + 1) / (n_perm + 1)
    p_fdr = fdrcorrection(p_raw)[1]
    p_use = p_fdr if fdr else p_raw

    results = pd.DataFrame({
        "network_1": [order[i] for i in rows], "network_2": [order[j] for j in cols],
        "observed": obs_vals.astype(int), "possible": possible[rows, cols].astype(int),
        "density": density[rows, cols], "p": p_raw, "p_fdr": p_fdr,
    }).sort_values("p").reset_index(drop=True)

    # display layout: optionally insert an empty Cerebellum row/column before Subcortical (plot only)
    disp = list(order)
    if collapse and pad_cerebellum and "Cerebellum" not in disp and "Subcortical" in disp:
        disp.insert(disp.index("Subcortical"), "Cerebellum")
    pos = np.array([disp.index(n) for n in order])          # order index -> display index
    n_disp = len(disp)
    density_disp = np.full((n_disp, n_disp), np.nan)
    density_disp[np.ix_(pos, pos)] = density
    shown = np.zeros((n_disp, n_disp), dtype=bool)
    shown[pos[rows], pos[cols]] = True
    pad_idx = [k for k, n in enumerate(disp) if n not in order]
    d_rows, d_cols = np.tril_indices(n_disp, k=0 if include_within else -1)
    pad_cells = [(i, j) for i, j in zip(d_rows, d_cols) if i in pad_idx or j in pad_idx]
    labels = [short_names.get(n, n) for n in disp]
    vmax = np.nanmax(density[rows, cols])
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    if cmap == "Greens":
        cmap = LinearSegmentedColormap.from_list("light_greens", plt.get_cmap("Greens")(np.linspace(0, 0.98, 256)))

    with plt.rc_context({"font.family": "Arial"}):
        size = 4 * n_disp / 7
        fig, ax = plt.subplots(figsize=(size, size))
        sns.heatmap(density_disp, mask=~shown, cmap=cmap, vmin=0, vmax=vmax, square=True,
                    linewidths=0.5, linecolor="black", xticklabels=labels, yticklabels=labels,
                    cbar_kws={"shrink": 0.5}, ax=ax)
        ax.set_xticklabels(labels, rotation=90, ha="center", size=14)
        ax.set_yticklabels(labels, rotation=0, va="center", size=14)
        ax.tick_params(axis="both", which="both", length=0)
        for i, j in pad_cells:                              # network not in this atlas: grey cell
            ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor="#a6a6a6", edgecolor="black", linewidth=0.5,
                                       zorder=2))
        ax.hlines(n_disp, 0, n_disp, colors="black", linewidth=2)
        ax.vlines(n_disp, 0, n_disp, colors="black", linewidth=2)
        cbar = ax.collections[0].colorbar
        cbar.set_ticks([0, vmax])
        cbar.set_ticklabels(["0", f"{vmax:.3f}"])
        cbar.ax.tick_params(labelsize=14)
        cbar.outline.set_visible(False)
        cbar.set_label("Proportion of\nEdges in Network", size=12)
        for i, j, p in zip(rows, cols, p_use):
            if p >= alpha:
                continue
            stars = "***" if p < 0.001 else "**" if p < 0.01 else "*"
            color = "white" if density[i, j] >= vmax / 1.5 else "black"   # white on dark cells, as in c_Network
            ax.text(pos[j] + 0.5, pos[i] + 0.64, stars, ha="center", va="center", color=color, fontsize=18, fontweight="bold")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return results, fig

In [ ]:
four_way_csv = os.path.join(NET_DIR, f"overlapping_features_{FOUR_WAY}.csv")
matrix_4way, fig = network_edge_matrix(four_way_csv, n_perm=N_PERM_NETWORK, fdr=FDR, seed=SEED, include_within=True,
                                       save_path=os.path.join(FIG_DIR, "4way_network_matrix.png"))
matrix_4way.to_csv(os.path.join(NET_DIR, "4way_network_matrix.csv"), index=False)
plt.show()
# Schaefer only: the same matrix for the 17 subnetworks
# matrix_4way_17, fig17 = network_edge_matrix(four_way_csv, collapse=False, n_perm=N_PERM_NETWORK, fdr=FDR, seed=SEED,
#                                             include_within=True,
#                                             save_path=os.path.join(FIG_DIR, "4way_network_matrix_17networks.png"))
# matrix_4way_17.to_csv(os.path.join(NET_DIR, "4way_network_matrix_17networks.csv"), index=False)
# plt.show()
matrix_4way

# Annotate ROI Overlap

In [ ]:
def label_features_shen(feature_csv, atlas_csv=ATLAS_CSV, out_csv=None):
    """Add ROI1_<col> / ROI2_<col> for every column of the atlas CSV (Network, ...) and save *_labeled.csv."""
    atlas = pd.read_csv(atlas_csv)
    atlas["ROI"] = atlas["ROI"].astype(int)
    info_cols = [c for c in atlas.columns if c != "ROI"]
    lookup = atlas.set_index("ROI")[info_cols]
    df = pd.read_csv(feature_csv)
    for col in ["ROI1", "ROI2"]:
        rois = df[col].astype(int)
        unknown = sorted(set(rois) - set(lookup.index))
        if unknown:
            print(f"{col}: no label for ROI number(s) {unknown}")
        for c in info_cols:
            df[f"{col}_{c.replace(' ', '_')}"] = rois.map(lookup[c]).values
    if out_csv is None:
        out_csv = str(feature_csv).rsplit(".csv", 1)[0] + "_labeled.csv"
    df.to_csv(out_csv, index=False)
    return df


# Display names for tables: Schaefer 2018 (17-network) parcel abbreviations -> anatomical region names
REGION_NAMES = {
    "ExStr": "extrastriate cortex", "ExStrInf": "inferior extrastriate cortex",
    "ExStrSup": "superior extrastriate cortex", "Striate": "striate cortex", "StriCal": "striate/calcarine cortex",
    "Aud": "auditory cortex", "Cent": "central sulcus", "S2": "secondary somatosensory cortex",
    "TempOcc": "temporo-occipital cortex", "ParOcc": "parieto-occipital cortex", "SPL": "superior parietal lobule",
    "FEF": "frontal eye field", "PostC": "postcentral gyrus", "PrC": "precentral cortex",
    "FrMed": "medial frontal cortex", "FrOper": "frontal operculum", "Ins": "insula",
    "ParMed": "medial parietal cortex", "ParOper": "parietal operculum",
    "IPL": "inferior parietal lobule", "IPS": "intraparietal sulcus", "Temp": "temporal cortex",
    "AntTemp": "anterior temporal cortex", "TempPole": "temporal pole", "OFC": "orbitofrontal cortex",
    "PFCl": "lateral prefrontal cortex", "PFClv": "ventrolateral prefrontal cortex",
    "PFCld": "dorsolateral prefrontal cortex", "PFCd": "dorsal prefrontal cortex",
    "PFCm": "medial prefrontal cortex", "PFCmp": "posterior medial prefrontal cortex",
    "PFCv": "ventral prefrontal cortex", "Cingm": "mid-cingulate cortex", "Cingp": "posterior cingulate cortex",
    "pCun": "precuneus", "pCunPCC": "precuneus/posterior cingulate cortex", "PHC": "parahippocampal cortex",
    "Rsp": "retrosplenial cortex",
}
# networks whose labels carry no region part ("17Networks_LH_SomMotA_3", "17Networks_RH_TempPar_2")
NETWORK_ONLY_REGION = {"SomMotA": "somatomotor cortex", "TempPar": "temporoparietal junction"}
HEMI = {"LH": "L", "RH": "R"}
_unknown_regions = set()


def roi_display(roi, lookup=LOOKUP):
    """(hemisphere, region, display name) for one ROI number, e.g. (L, lateral prefrontal cortex,
    'L lateral prefrontal cortex'). Subcortical ROIs use SUBCORTICAL_NAMES (or 'Subcortical k')."""
    if roi not in lookup:
        return None, None, None
    label, sub, parent = lookup[roi]
    if sub == "Subcortical":
        name = label.replace("_", " ")
        return "", name, name
    parts = label.split("_")                      # 17Networks, LH, ContB, PFCl, 1  (region part may be absent)
    hemi = HEMI.get(parts[1], parts[1])
    abbr = parts[3] if len(parts) > 4 else None
    if abbr is None:
        region = NETWORK_ONLY_REGION.get(sub, sub)
    else:
        region = REGION_NAMES.get(abbr)
        if region is None:
            _unknown_regions.add(abbr)
            region = abbr
    return hemi, region, f"{hemi} {region}"


def label_features(feature_csv, lookup=LOOKUP, out_csv=None):
    """Add <col>_name, <col>_network17, <col>_network (as before) plus display columns for tables:
    <col>_hemisphere, <col>_region, <col>_display ('L lateral prefrontal cortex') and
    <col>_display_network ('L lateral prefrontal cortex (CON-B)'), and an 'Edge' column. Saves *_labeled.csv."""
    df = pd.read_csv(feature_csv)
    for col in ["ROI1", "ROI2"]:
        rois = df[col].astype(int)
        unknown = sorted(set(rois) - set(lookup))
        if unknown:
            print(f"{col}: no label for ROI number(s) {unknown}")
        df[f"{col}_name"] = rois.map(lambda r: lookup.get(r, (None,) * 3)[0])
        df[f"{col}_network17"] = rois.map(lambda r: lookup.get(r, (None,) * 3)[1])
        df[f"{col}_network"] = rois.map(lambda r: lookup.get(r, (None,) * 3)[2])
        disp = rois.map(lambda r: roi_display(r, lookup))
        df[f"{col}_hemisphere"] = disp.map(lambda d: d[0])
        df[f"{col}_region"] = disp.map(lambda d: d[1])
        df[f"{col}_display"] = disp.map(lambda d: d[2])
        df[f"{col}_display_network"] = [f"{d} ({SHORT17.get(n, n)})" if d else None
                                       for d, n in zip(df[f"{col}_display"], df[f"{col}_network17"])]
    df["Edge"] = df["ROI1_display"] + " – " + df["ROI2_display"]
    if out_csv is None:
        out_csv = str(feature_csv).rsplit(".csv", 1)[0] + "_labeled.csv"
    df.to_csv(out_csv, index=False)
    return df


def edge_table(labeled, out_csv=None):
    """Publication-style table of an overlap: one row per edge, in the same order as the other exports
    (ROI1, then ROI2)."""
    t = pd.DataFrame({
        "Sign": labeled["Sign"].map({"pos": "+", "neg": "−"}) if "Sign" in labeled else "",
        "Region 1": labeled["ROI1_display"], "Network 1": labeled["ROI1_network17"].map(lambda n: SHORT17.get(n, n)),
        "Region 2": labeled["ROI2_display"], "Network 2": labeled["ROI2_network17"].map(lambda n: SHORT17.get(n, n)),
        "ROI 1": labeled["ROI1"], "ROI 2": labeled["ROI2"],
    }).sort_values(["ROI 1", "ROI 2"]).reset_index(drop=True)
    t.index = t.index + 1
    if out_csv:
        t.to_csv(out_csv, index_label="#")
    return t


label_fn = label_features_shen if ATLAS == "Shen" else label_features
for _f in sorted(os.listdir(NET_DIR)):
    if _f.startswith("overlapping_features_") and _f.endswith(".csv") and not _f.endswith(("_labeled.csv", "_table.csv")):
        _df = label_fn(os.path.join(NET_DIR, _f))
        print(f"labeled {len(_df):5d} edges  →  {_f[:-4]}_labeled.csv")
labeled_4way = pd.read_csv(os.path.join(NET_DIR, f"overlapping_features_{FOUR_WAY}_labeled.csv"))
if ATLAS == "Schaefer":                                # region display names + a publication-style edge table
    if _unknown_regions:
        print(f"⚠️ No display name for region abbreviation(s) {sorted(_unknown_regions)}: add them to REGION_NAMES")
    table_4way = edge_table(labeled_4way, os.path.join(NET_DIR, f"overlapping_features_{FOUR_WAY}_table.csv"))
    print(f"Saved overlapping_features_{FOUR_WAY}_table.csv ({len(table_4way)} edges)")
labeled_4way.head()

# Predict Arousal from 4way Network

In [ ]:
# Uses only the edges that all four models selected with the same sign (+1 / -1 weights) and predicts each
# dataset's arousal like the cross-dataset models in b_Classification. Results go to
# <classification folder>/<number>_<Dataset>_from_Network/ and the figure to <classification folder>/figures/.
from scipy.stats import pearsonr
from numpy.fft import fft, ifft

# False = the 4-way network for every dataset (as requested).
# True  = for each dataset, use the 3-way overlap of the OTHER three models, so the dataset being predicted
#         played no part in choosing the edges (written to ..._from_NetworkHeldOut instead).
HOLD_OUT_TARGET = False
N_PERM_PRED     = 10000
NET_LABEL       = "NetworkHeldOut" if HOLD_OUT_TARGET else "Network"
PERM_BASE       = f"model5_p{f'{ALPHA:g}'.replace('0.', '', 1)}_mean"      # same file names as b_Classification
CLS_FIG_DIR     = os.path.join(CLS_DIR, "figures")

# cut FC / arousal files written by b_Classification (Shen has no suffix; Schaefer files end in _Schaefer)
_SFX = "" if ATLAS == "Shen" else f"_{ATLAS}"
CUT_FILES = {
    "EEG_Rest":    ("1_EEG_Rest/EEG_Rest_FC_Cut", "1_EEG_Rest/EEG_Rest_Arousal_Cut"),
    "Pupil_Rest":  ("2_Pupil_Rest/Pupil_Rest_FC_Cut", "2_Pupil_Rest/Pupil_Rest_Arousal_Cut"),
    "Pupil_Story": ("3_Pupil_Story/Pupil_Story_FC_Cut", "3_Pupil_Story/Pupil_Story_Arousal_Cut"),
    "FilmFest":    ("4_FilmFestival/FilmFestival_FC_Cut", "4_FilmFestival/FilmFestival_Arousal_Cut"),
    "Sherlock":    ("5_Sherlock/Sherlock_FC_Cut", "5_Sherlock/Sherlock_Arousal_Cut"),
}
# (folder number, dataset, plot label, model name in FEATURE_FILES, colour)
NET_TARGETS = [(20, "Pupil_Rest", "Pupil Rest", "PupilRest", "#e41a1c"),
               (21, "EEG_Rest", "EEG Rest", "EEGRest", "#377eb8"),
               (22, "Pupil_Story", "Pupil Story", "PupilStory", "#4daf4a"),
               (23, "Movies", "Movies", "Movies", "#984ea3")]


def load_cut(name):
    fc_rel, ar_rel = CUT_FILES[name]
    fc = np.load(os.path.join(RESULTS, "1_arousal", f"{fc_rel}{_SFX}.npy"), mmap_mode="r")
    ar = np.load(os.path.join(RESULTS, "1_arousal", f"{ar_rel}{_SFX}.npy"))
    return fc, ar


def network_model(model, n_features):
    """Edges + signs of the network used to predict `model`'s dataset, as b_Classification models (sig, signs)."""
    if HOLD_OUT_TARGET:
        combo = "_".join(m for m in FEATURE_FILES if m != model)
    else:
        combo = FOUR_WAY
    edges = pd.read_csv(os.path.join(NET_DIR, f"overlapping_features_{combo}.csv"))
    n_rois = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
    pair_index = {p: i for i, p in enumerate(combinations(range(1, n_rois + 1), 2))}
    idx = np.array([pair_index[(min(a, b), max(a, b))] for a, b in zip(edges["ROI1"].astype(int), edges["ROI2"].astype(int))],
                   dtype=int)
    signs = np.where(edges["Sign"].astype(str).str.lower() == "neg", -1.0, 1.0)
    order = np.argsort(idx)                         # ascending feature index, like np.where in fit_model
    sig, signs = idx[order], signs[order]
    feats = pd.DataFrame({"ROI1": edges["ROI1"].astype(int).to_numpy()[order],
                          "ROI2": edges["ROI2"].astype(int).to_numpy()[order],
                          "feature": sig, "feature_weight": signs})
    return sig, signs, feats, combo


def predict_cross(fc_target, sig, weights, total_features):
    """(Σ fc·w over w>0 + Σ fc·w over w<0) / n_sig — identical to b_Classification."""
    n_subj, n_trs = fc_target.shape[:2]
    pred = np.full((n_subj, n_trs), np.nan)
    pos, neg = weights > 0, weights < 0
    divisor = total_features if total_features > 0 else 1
    for s in range(n_subj):
        masked = np.asarray(fc_target[s])[:, sig] * weights
        for i in range(n_trs):
            v = masked[i, :]
            pred[s, i] = np.sum(v[pos]) / divisor + np.sum(v[neg]) / divisor
    return pred


# ---- permutation statistics (same as b_Classification) ----
def fisher_z(r):
    r = np.clip(r, -0.999999, 0.999999)
    return 0.5 * (np.log1p(r) - np.log1p(-r))


def fisher_z_inv(z):
    return (np.exp(2 * z) - 1) / (np.exp(2 * z) + 1)


def subject_rvals(pred, obs):
    """Per-subject Pearson r on TRs where both are present (compute_observed_value in b_Classification)."""
    n = min(pred.shape[0], obs.shape[0])
    out = []
    for s in range(n):
        ok = ~np.isnan(pred[s]) & ~np.isnan(obs[s])
        if ok.sum() > 1:
            out.append(pearsonr(pred[s][ok], obs[s][ok])[0])
    return np.array(out)


def _phase_freqs(n):
    if n % 2 == 0:
        return np.arange(1, n // 2), np.arange(n - 1, n // 2, -1)
    return np.arange(1, (n - 1) // 2 + 1), np.arange(n - 1, (n - 1) // 2, -1)


def _phase_randomize_batch(obs, phases):
    pos_freq, neg_freq = _phase_freqs(obs.shape[-1])
    if len(pos_freq) == 0:
        return np.real(ifft(np.broadcast_to(fft(obs, axis=-1), phases.shape[:2] + obs.shape[-1:]), axis=-1))
    pos = slice(pos_freq[0], pos_freq[-1] + 1)
    neg = slice(neg_freq[0], neg_freq[-1] - 1, -1)
    F = np.broadcast_to(fft(obs, axis=-1), phases.shape[:2] + obs.shape[-1:]).copy()
    E = np.exp(1j * phases)
    F[..., pos] *= E
    F[..., neg] *= E.conj()
    return np.real(ifft(F, axis=-1))


def _draw_phases(rs, B, groups):
    sizes = [n * len(_phase_freqs(T)[0]) for n, T in groups]
    flat = rs.uniform(0, 2 * np.pi, size=B * sum(sizes)).reshape(B, sum(sizes))
    out_, start = [], 0
    for (n, T), sz in zip(groups, sizes):
        out_.append(flat[:, start:start + sz].reshape(B, n, sz // n))
        start += sz
    return out_


def _rowwise_r(pred, surr, masked):
    P = np.broadcast_to(pred, surr.shape)
    if masked and not (np.isnan(pred).any() or np.isnan(surr).any()):
        masked = False
    valid = (~np.isnan(P) & ~np.isnan(surr)) if masked else np.ones(surr.shape, dtype=bool)
    n = valid.sum(-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        x = np.where(valid, P, 0.0) if masked else P
        y = np.where(valid, surr, 0.0) if masked else surr
        xm = x - x.sum(-1, keepdims=True) / n[..., None]
        ym = y - y.sum(-1, keepdims=True) / n[..., None]
        if masked:
            xm, ym = np.where(valid, xm, 0.0), np.where(valid, ym, 0.0)
        r = (xm * ym).sum(-1) / np.sqrt((xm * xm).sum(-1) * (ym * ym).sum(-1))
    return np.clip(r, -1.0, 1.0), n


def _group_mean_r(r, include):
    z = np.where(include, fisher_z(r), 0.0)
    return fisher_z_inv(z.sum(-1) / include.sum(-1))


def _batches(n_perm, per_perm_elems, budget=4_000_000):
    B = max(1, int(budget // max(per_perm_elems, 1)))
    for s in range(0, n_perm, B):
        yield min(B, n_perm - s)


def null_single(pred, obs, n_perm, rs):
    """Phase-randomised null for one dataset (masked, like permutation_test in b_Classification)."""
    n_min = min(pred.shape[0], obs.shape[0])
    null = []
    for B in _batches(n_perm, obs.size):
        (ph,) = _draw_phases(rs, B, [obs.shape])
        r, n = _rowwise_r(pred[:n_min], _phase_randomize_batch(obs, ph)[:, :n_min], masked=True)
        null.append(_group_mean_r(r, n > 1))
    return np.concatenate(null)


def null_movies(pred_ff, obs_ff, pred_s, obs_s, n_perm, rs):
    """FilmFest + Sherlock pooled (like permutation_test_toSR in b_Classification)."""
    null = []
    for B in _batches(n_perm, obs_ff.size + obs_s.size):
        ph_ff, ph_s = _draw_phases(rs, B, [obs_ff.shape, obs_s.shape])
        ra, _ = _rowwise_r(pred_ff, _phase_randomize_batch(obs_ff, ph_ff), masked=False)
        rb, _ = _rowwise_r(pred_s, _phase_randomize_batch(obs_s, ph_s), masked=False)
        r = np.concatenate([ra, rb], axis=1)
        null.append(_group_mean_r(r, np.ones(r.shape, dtype=bool)))
    return np.concatenate(null)


def save_perm(out_dir, rvals, null, n_perm):
    observed = fisher_z_inv(np.mean(fisher_z(rvals)))
    p = (np.sum(null >= observed) + 1) / (n_perm + 1)
    base = os.path.join(out_dir, PERM_BASE)
    np.save(base + "_null_distribution.npy", null)
    np.save(base + "_observed_rvals.npy", rvals)
    with open(base + "_pval.txt", "w") as f:
        f.write(f"{p:.6f}")
    pd.DataFrame({"subject": np.arange(len(rvals)), "pearson_r": rvals}).to_csv(base + "_subject_rvals.csv", index=False)
    with open(base + "_mean_observed_r.txt", "w") as f:
        f.write(f"{observed:.6f}")
    return observed, p

In [ ]:
rs = np.random.RandomState(SEED)
NET_RESULTS = {}
for idx, target, label, model, colour in NET_TARGETS:
    folder = f"{idx}_{target}_from_{NET_LABEL}"
    out_dir = os.path.join(CLS_DIR, folder)
    os.makedirs(out_dir, exist_ok=True)
    first = "FilmFest" if target == "Movies" else target
    n_features = np.load(os.path.join(RESULTS, "1_arousal", f"{CUT_FILES[first][0]}{_SFX}.npy"), mmap_mode="r").shape[2]
    sig, w, feats, combo = network_model(model, n_features)
    if len(sig) == 0:
        print(f"{folder}: the {combo} network has no edges → skipped")
        continue

    if target == "Movies":
        fc_ff, ar_ff = load_cut("FilmFest")
        fc_s, ar_s = load_cut("Sherlock")
        pred_ff = predict_cross(fc_ff, sig, w, len(sig))
        pred_s = predict_cross(fc_s, sig, w, len(sig))
        for d, pred, obs in [("FilmFest", pred_ff, ar_ff), ("Sherlock", pred_s, ar_s)]:
            np.save(os.path.join(out_dir, f"{d}_from_{NET_LABEL}_predicted.npy"), pred)
            np.save(os.path.join(out_dir, f"{d}_from_{NET_LABEL}_observed.npy"), obs)
        feats.to_csv(os.path.join(out_dir, f"Movies_from_{NET_LABEL}_features.csv"), index=False)
        rvals = np.array([pearsonr(pred_ff[i], ar_ff[i])[0] for i in range(pred_ff.shape[0])]
                         + [pearsonr(pred_s[i], ar_s[i])[0] for i in range(pred_s.shape[0])])
        null = null_movies(pred_ff, ar_ff, pred_s, ar_s, N_PERM_PRED, rs)
    else:
        fc, ar = load_cut(target)
        pred = predict_cross(fc, sig, w, len(sig))
        np.save(os.path.join(out_dir, f"{target}_from_{NET_LABEL}_predicted.npy"), pred)
        np.save(os.path.join(out_dir, f"{target}_from_{NET_LABEL}_observed.npy"), ar)
        feats.to_csv(os.path.join(out_dir, f"{target}_from_{NET_LABEL}_features.csv"), index=False)
        rvals = subject_rvals(pred, ar)
        null = null_single(pred, ar, N_PERM_PRED, rs)

    mean_r, p = save_perm(out_dir, rvals, null, N_PERM_PRED)
    NET_RESULTS[label] = {"rvals": rvals, "null": null, "pval": p, "colour": colour}
    print(f"{folder:36s} {len(sig):4d} edges ({combo})   mean r = {mean_r:.3f}   p = {p:.4f}")